In [29]:
!pip install -q langchain-groq groq

In [30]:
import os
from google.colab import userdata

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    ToolMessage
)

In [31]:
api_key = userdata.get("GROQ_API_KEY")

os.environ["GROQ_API_KEY"] = api_key

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [35]:
restaurant_menu = {
    "pizza": {
        "price": 250,
        "available": True
    },

    "burger": {
        "price": 120,
        "available": True
    },

    "pasta": {
        "price": 180,
        "available": True
    },

    "biryani": {
        "price": 200,
        "available": True
    },

    "french fries": {
        "price": 100,
        "available": True
    },

    "sandwich": {
        "price": 90,
        "available": False
    }
}

current_order = {}
placed_orders = []
order_counter = 1001

confirmation_pending = False

print("Restaurant menu loaded successfully!")

Restaurant menu loaded successfully!


In [36]:
@tool
def view_menu():
    """Display all restaurant menu items, prices and availability."""
    result = "Restaurant Menu:\n"

    for item, details in restaurant_menu.items():
        status = "Available" if details["available"] else "Not Available"

        result += (
            f"{item.title()} - Rs.{details['price']} - {status}\n"
        )

    return result


@tool
def check_availability(item: str):
    """Check whether a particular food item is available."""
    item = item.lower().strip()

    if item not in restaurant_menu:
        return "Sorry, this item is not present in our menu."

    if restaurant_menu[item]["available"]:
        return f"{item.title()} is available."

    return f"Sorry, {item.title()} is currently unavailable."


@tool
def add_to_order(item: str, quantity: int):
    """Add a food item and quantity to the customer's order."""
    item = item.lower().strip()

    if item not in restaurant_menu:
        return "This item is not present in our menu."

    if not restaurant_menu[item]["available"]:
        return f"Sorry, {item.title()} is unavailable."

    if quantity <= 0:
        return "Quantity must be greater than zero."

    current_order[item] = current_order.get(item, 0) + quantity

    return f"{quantity} {item}(s) added to your order."


@tool
def view_order():
    """Display all items currently added to the customer's order."""
    if not current_order:
        return "Your order is empty."

    result = "Your Current Order:\n"

    for item, quantity in current_order.items():
        price = restaurant_menu[item]["price"]
        result += (
            f"{item.title()} x {quantity} = Rs.{price * quantity}\n"
        )

    return result


@tool
def calculate_bill():
    """Calculate subtotal, 5 percent tax and final bill."""
    global confirmation_pending

    if not current_order:
        return "Your order is empty. Please add items first."

    subtotal = 0

    for item, quantity in current_order.items():
        subtotal += restaurant_menu[item]["price"] * quantity

    tax = subtotal * 0.05
    total = subtotal + tax

    confirmation_pending = True

    return (
        f"Subtotal: Rs.{subtotal}\n"
        f"Tax (5%): Rs.{tax:.2f}\n"
        f"Total Bill: Rs.{total:.2f}\n"
        "Please confirm your order by replying YES."
    )


@tool
def place_order():
    """Place the order only after the customer explicitly confirms it."""
    global order_counter
    global confirmation_pending
    global user_confirmed

    if not current_order:
        return "Your order is empty. Please add items first."

    if not confirmation_pending or not user_confirmed:
        return "Please review your bill and explicitly confirm by replying YES."

    order = {
        "order_id": order_counter,
        "items": current_order.copy()
    }

    placed_orders.append(order)

    confirmation_pending = False
    user_confirmed = False
    current_order.clear()

    order_counter += 1

    return f"Your order has been placed successfully! Order ID: {order['order_id']}"


user_confirmed = False

print("Restaurant tools created successfully!")

Restaurant tools created successfully!


In [37]:
tools = [
    view_menu,
    check_availability,
    add_to_order,
    view_order,
    calculate_bill,
    place_order
]

tool_map = {
    selected_tool.name: selected_tool
    for selected_tool in tools
}

llm_with_tools = llm.bind_tools(tools)

print("All tools registered successfully!")

All tools registered successfully!


In [38]:
SYSTEM = """
You are a helpful Restaurant Order Management AI Agent.

Your responsibilities:

1. Help customers view the restaurant menu.
2. Check food item availability.
3. Add food items to customer orders.
4. Show the current order.
5. Calculate the bill.
6. Place the order only after explicit customer confirmation.

Rules:

- Always use the tools for menu, availability, order and bill details.
- Never invent food items or prices.
- Never add unavailable items.
- Ask for quantity if the customer does not mention it.
- Before placing an order, show the bill and ask for confirmation.
- Never place an order without explicit YES confirmation.
- Be polite and give short, clear responses.
"""

print("System prompt created successfully!")

System prompt created successfully!


In [40]:
messages = [
    SystemMessage(content=SYSTEM)
]

def restaurant_agent(user_input):

    global user_confirmed

    confirmation_words = [
        "yes",
        "yes please",
        "confirm",
        "confirm order",
        "yes, confirm order",
        "go ahead"
    ]

    user_confirmed = user_input.strip().lower() in confirmation_words

    messages.append(HumanMessage(content=user_input))

    for step in range(8):

        response = llm_with_tools.invoke(messages)

        messages.append(response)

        if not response.tool_calls:
            return response.content

        for tool_call in response.tool_calls:

            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            selected_tool = tool_map[tool_name]

            result = selected_tool.invoke(tool_args)

            print("Tool Used:", tool_name)
            print("Tool Result:", result)
            print("-------------------------")

            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )

    return "Please try your request again."

In [41]:
restaurant_agent("Show me the restaurant menu")

Tool Used: view_menu
Tool Result: Restaurant Menu:
Pizza - Rs.250 - Available
Burger - Rs.120 - Available
Pasta - Rs.180 - Available
Biryani - Rs.200 - Available
French Fries - Rs.100 - Available
Sandwich - Rs.90 - Not Available

-------------------------


'Here’s our current menu:\n\n- **Pizza** – Rs.250 – Available  \n- **Burger** – Rs.120 – Available  \n- **Pasta** – Rs.180 – Available  \n- **Biryani** – Rs.200 – Available  \n- **French Fries** – Rs.100 – Available  \n- **Sandwich** – Rs.90 – *Not Available*\n\nLet me know what you’d like to add to your order (and the quantity, if you have a preference).'

In [42]:
restaurant_agent("Is pizza available?")

Tool Used: check_availability
Tool Result: Pizza is available.
-------------------------


'Yes, pizza is currently available. Would you like to add any to your order? If so, please tell me the quantity.'

In [43]:
restaurant_agent("I want 2 pizzas and 1 burger")

Tool Used: add_to_order
Tool Result: 2 pizza(s) added to your order.
-------------------------
Tool Used: add_to_order
Tool Result: 1 burger(s) added to your order.
-------------------------
Tool Used: view_order
Tool Result: Your Current Order:
Pizza x 2 = Rs.500
Burger x 1 = Rs.120

-------------------------
Tool Used: calculate_bill
Tool Result: Subtotal: Rs.620
Tax (5%): Rs.31.00
Total Bill: Rs.651.00
Please confirm your order by replying YES.
-------------------------


'Your order summary:\n\n- Pizza x\u202f2 = Rs.500  \n- Burger x\u202f1 = Rs.120  \n\n**Subtotal:** Rs.620  \n**Tax (5%):** Rs.31.00  \n**Total:** Rs.651.00  \n\nPlease reply **YES** to place the order, or let me know if you’d like any changes.'

In [44]:
restaurant_agent("Show my current order")

Tool Used: view_order
Tool Result: Your Current Order:
Pizza x 2 = Rs.500
Burger x 1 = Rs.120

-------------------------


'Here’s your current order:\n\n- Pizza\u202f×\u202f2 = Rs.500  \n- Burger\u202f×\u202f1 = Rs.120  \n\nLet me know if you’d like to add anything else, modify quantities, or confirm the order.'

In [45]:
restaurant_agent("Calculate my total bill")

Tool Used: calculate_bill
Tool Result: Subtotal: Rs.620
Tax (5%): Rs.31.00
Total Bill: Rs.651.00
Please confirm your order by replying YES.
-------------------------


'**Bill Summary**\n\n- **Subtotal:** Rs.620  \n- **Tax (5%):** Rs.31.00  \n- **Total:** Rs.651.00  \n\nPlease reply **YES** to place the order, or let me know if you’d like any changes.'

In [46]:
restaurant_agent("Yes")

Tool Used: place_order
Tool Result: Your order has been placed successfully! Order ID: 1001
-------------------------


'Your order is confirmed! 🎉  \n**Order ID:** 1001  \n**Total Paid:** Rs.651.00  \n\nThank you for ordering with us. Enjoy your meal!'